<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/New-Baseline-v2/mnps_new_baseline%20v7.5.3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

# MNPS Job Classification - Qwen2.5 (Two-Pass Self-Consistency)

**Version 7.5.5 - IMPROVED FOR COLAB WITH MODEL SELECTION**

## Choose Your Model (Section 7):
- **Qwen2.5-7B** - Fast, good quality (7-8 GB) - Default
- **Qwen2.5-14B** - Best balance (12-14 GB) - ⭐ Recommended
- **Qwen2.5-32B** - Excellent quality (28-30 GB) - Slower

## Changes in this version:
- ✅ **Easy model switching** - Just uncomment one line!
- ✅ Fixed memory management for A100 GPU
- ✅ Added proper 4-bit quantization
- ✅ Improved error handling
- ✅ Added GPU memory monitoring
- ✅ Fixed missing imports
- ✅ Optimized batch processing

**Recommended:** Use Qwen2.5-14B-Instruct for best accuracy/speed balance


## 1) Setup: Install Libraries

In [ ]:
# Install required libraries with specific versions for stability
!pip install -q "transformers>=4.41.0,<5.0.0" accelerate bitsandbytes safetensors torch tqdm
print("✅ Libraries installed successfully")

## 2) Check GPU and Memory

In [ ]:
import torch
import os

# Check GPU availability
if torch.cuda.is_available():
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("❌ No GPU detected! This notebook requires GPU runtime.")
    print("   Please go to Runtime > Change runtime type > Hardware accelerator > GPU (A100)")

## 3) Mount Drive and Setup Folders

In [ ]:
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from tqdm.auto import tqdm
from google.colab import drive

# Import for Qwen model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import gc

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}_Qwen"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

## 4) Load Data Files

In [ ]:
# Load all required files
RUN_ROOT = Path('/content')

# Unzip MNPS Prompt Resources if needed
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"
if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found - make sure to upload it")

# Core data files
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

# If main input file not found, check inside the zip
if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside MNPS Prompt Resources.zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            print("❌ Sample JDs.csv not found in zip contents:", zip_contents)
            raise FileNotFoundError("Sample JDs.csv not found in root or zip")
    # Optionally extract Ground Truth if present
    if "Ground Truth Masterfile.csv" in zip_contents:
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extract("Ground Truth Masterfile.csv", RUN_ROOT)
            print("✅ Extracted Ground Truth Masterfile.csv from zip")

# MNPS Prompt Resources (from extracted zip)
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

print(f"📄 Batch input: {BATCH_INPUT_CSV}")
print(f"📄 Ground truth: {GT_MASTERFILE_CSV}")
print(f"📄 MNPS roles: {MNPS_ROLES_CSV}")
print(f"📄 MNPS KSACs: {MNPS_KSACS_CSV}")
print(f"📄 Competency Extended: {COMPETENCY_EXTENDED_CSV}")
print(f"📄 Korn Ferry: {KORN_FERRY_CSV}")

# Load data
try:
    df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
    gt_df = pd.read_csv(GT_MASTERFILE_CSV, encoding='latin1')
    roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
    ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
    competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
    korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')
    
    print(f"✅ Loaded {len(df)} job descriptions")
    print(f"✅ Loaded {len(gt_df)} ground truth records")
    print(f"✅ Loaded {len(roles_df)} MNPS roles")
    print(f"✅ Loaded {len(ksacs_df)} MNPS KSACs")
    print(f"✅ Loaded {len(competency_df)} competency descriptions")
    print(f"✅ Loaded {len(korn_ferry_df)} Korn Ferry competencies")
except Exception as e:
    print(f"❌ Error loading data files: {e}")
    raise

## 5) Build Attribute-Only View (Ignore Title)

In [ ]:
# Load prediction data (if available from previous runs)
preds = df.copy()

# Build attribute-only text (ignore job titles)
ATTR_COLS = [
    'Position Summary', 'Essential Functions', 'Work Experience', 'Education',
    'Licenses and Certifications', 'Knowledge, Skills and Abilities'
]

# Combine all attribute text
attrs = df[ATTR_COLS].fillna('')
text = attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' + \
       attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' + \
       attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities']

print(f"✅ Built attribute-only view for {len(text)} job descriptions")
print(f"✅ Ignoring job titles - focusing on job attributes only")

## 6) Setup Closed Sets and Normalization

In [ ]:
# Get MNPS roles from the loaded data
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
if role_columns:
    MNPS_ROLES = sorted(roles_df[role_columns[0]].dropna().unique().tolist())
else:
    # Fallback to first column if no 'role' column found
    MNPS_ROLES = sorted(roles_df.iloc[:, 0].dropna().unique().tolist())

print(f"✅ Loaded {len(MNPS_ROLES)} MNPS roles")

# Define other role categories
MINOR_SUB_GROUPS = ['I', 'II', 'III', 'IV', 'Lead']
EXECUTIVE_ROLES = ['Executive', 'Director', 'Chief']

# Normalization dictionaries
MAJOR_ROLE_ALIASES = {
    'specialist': 'Specialist',
    'teacher': 'Teacher',
    'support staff': 'Support Staff',
    'administrator': 'Administrator',
    'other': 'Other'
}

MINOR_ROLE_ALIASES = {
    'i': 'I',
    'ii': 'II',
    'iii': 'III',
    'iv': 'IV',
    'lead': 'Lead',
    '1': 'I',
    '2': 'II',
    '3': 'III',
    '4': 'IV'
}

def normalize_major_role(role):
    """Normalize major role to match MNPS roles"""
    if pd.isna(role):
        return 'Other'
    
    role_str = str(role).strip()
    role_lower = role_str.lower()
    
    # Check aliases
    if role_lower in MAJOR_ROLE_ALIASES:
        return MAJOR_ROLE_ALIASES[role_lower]
    
    # Check if it matches any MNPS role
    for mnps_role in MNPS_ROLES:
        if role_lower == mnps_role.lower():
            return mnps_role
    
    return 'Other'

def normalize_minor_role(role):
    """Normalize minor role to valid sub-groups"""
    if pd.isna(role):
        return 'I'
    
    role_str = str(role).strip()
    role_lower = role_str.lower()
    
    # Check aliases
    if role_lower in MINOR_ROLE_ALIASES:
        return MINOR_ROLE_ALIASES[role_lower]
    
    # Check if it's already valid
    if role_str in MINOR_SUB_GROUPS:
        return role_str
    
    return 'I'

print("✅ Normalization functions defined")

## 7) Model Selection & Loading

In [ ]:
# ===== MODEL SELECTION =====
# Uncomment ONE line to choose your model:

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"      # Fast, good quality (7-8 GB)
# MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"    # RECOMMENDED - Best balance (12-14 GB)
# MODEL_NAME = "Qwen/Qwen2.5-32B-Instruct"    # Slower, excellent quality (28-30 GB)

# Model information
model_info = {
    "Qwen/Qwen2.5-7B-Instruct": {
        "memory": "7-8 GB",
        "speed": "Fast (3-8 sec/job)",
        "quality": "Good"
    },
    "Qwen/Qwen2.5-14B-Instruct": {
        "memory": "12-14 GB",
        "speed": "Medium (5-12 sec/job)",
        "quality": "Better (+10-15% vs 7B)"
    },
    "Qwen/Qwen2.5-32B-Instruct": {
        "memory": "28-30 GB",
        "speed": "Slower (8-20 sec/job)",
        "quality": "Excellent (+15-20% vs 7B)"
    }
}

info = model_info.get(MODEL_NAME, {})
print(f"📝 Selected Model: {MODEL_NAME}")
print(f"   Expected Memory: {info.get('memory', 'Unknown')}")
print(f"   Speed: {info.get('speed', 'Unknown')}")
print(f"   Quality: {info.get('quality', 'Unknown')}")
print("\n🤖 Loading model...")
print("   This may take a few minutes on first run...")

# Configure 4-bit quantization for memory efficiency
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

try:
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True
    )
    
    # Load model with quantization
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True
    )
    
    print("✅ Model loaded successfully!")
    print(f"   Model device: {model.device}")
    
    # Check memory usage
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"   GPU Memory: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")
    
except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

## 8) Helper Functions for Model Inference

In [ ]:
def generate_response(prompt, max_new_tokens=1024, temperature=0.7):
    """
    Generate response from Qwen model with proper error handling
    """
    try:
        # Prepare messages in chat format
        messages = [
            {"role": "system", "content": "You are an expert HR classifier for educational institutions."},
            {"role": "user", "content": prompt}
        ]
        
        # Apply chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        # Tokenize
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        # Generate
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Decode response
        response = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)
        
        # Clean up GPU memory
        del inputs, outputs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        return response.strip()
        
    except Exception as e:
        print(f"Error in generate_response: {e}")
        return f"ERROR: {str(e)}"

def parse_classification_response(response):
    """
    Parse the model's response to extract classification components
    """
    result = {
        'new_job_title': '',
        'major_role_group': 'Other',
        'minor_sub_group': 'I',
        'grouping_justification': response
    }
    
    try:
        # Extract job title
        title_match = re.search(r'Job Title[:\s]+([^\n]+)', response, re.IGNORECASE)
        if title_match:
            result['new_job_title'] = title_match.group(1).strip()
        
        # Extract major role
        major_match = re.search(r'Major Role[:\s]+([^\n]+)', response, re.IGNORECASE)
        if major_match:
            result['major_role_group'] = normalize_major_role(major_match.group(1).strip())
        
        # Extract minor role
        minor_match = re.search(r'Minor (?:Sub-?)?Group[:\s]+([^\n]+)', response, re.IGNORECASE)
        if minor_match:
            result['minor_sub_group'] = normalize_minor_role(minor_match.group(1).strip())
        
        # Extract justification
        just_match = re.search(r'Justification[:\s]+(.+)', response, re.IGNORECASE | re.DOTALL)
        if just_match:
            result['grouping_justification'] = just_match.group(1).strip()
    
    except Exception as e:
        print(f"Error parsing response: {e}")
    
    return result

print("✅ Helper functions defined")

## 9) Build Classification Prompts

In [ ]:
def build_classification_prompt(job_desc, roles_list, minor_groups):
    """
    Build the initial classification prompt
    """
    prompt = f"""You are an expert HR classifier for Metro Nashville Public Schools (MNPS).

Task: Classify the following job description into:
1. A new job title
2. A major role group
3. A minor sub-group

Available Major Role Groups:
{', '.join(roles_list)}

Available Minor Sub-Groups:
{', '.join(minor_groups)}

Job Description:
{job_desc}

Provide your classification in the following format:
Job Title: [suggested title]
Major Role Group: [selected from available options]
Minor Sub-Group: [I, II, III, IV, or Lead]
Justification: [explain your classification reasoning]
"""
    return prompt

def build_self_consistency_prompt(original_response, job_desc):
    """
    Build the self-consistency check prompt
    """
    prompt = f"""Review your previous classification for consistency.

Original Job Description:
{job_desc}

Your Previous Classification:
{original_response}

Task: Check if your justification aligns with the major role group and minor sub-group you selected.
If there's a mismatch, provide a corrected classification.

Respond with either:
1. "CONSISTENT - No changes needed" if the classification is correct
2. A corrected classification in the same format if changes are needed
"""
    return prompt

print("✅ Prompt templates defined")

## 10) Process Job Descriptions (Two-Pass)

In [ ]:
def process_job_description(row_idx, row):
    """
    Process a single job description with two-pass self-consistency check
    """
    try:
        # Build job description from attributes (ignore title)
        job_attrs = []
        for col in ATTR_COLS:
            if col in row and pd.notna(row[col]):
                job_attrs.append(f"{col}: {row[col]}")
        
        job_desc = "\n".join(job_attrs)
        
        # Pass 1: Initial classification
        prompt1 = build_classification_prompt(job_desc, MNPS_ROLES, MINOR_SUB_GROUPS)
        response1 = generate_response(prompt1, max_new_tokens=512)
        
        # Pass 2: Self-consistency check
        prompt2 = build_self_consistency_prompt(response1, job_desc)
        response2 = generate_response(prompt2, max_new_tokens=512)
        
        # Determine final response
        if "CONSISTENT" in response2.upper():
            final_response = response1
        else:
            final_response = response2
        
        # Parse the classification
        parsed = parse_classification_response(final_response)
        
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': parsed['new_job_title'],
            'major_role_group': parsed['major_role_group'],
            'minor_sub_group': parsed['minor_sub_group'],
            'grouping_justification': parsed['grouping_justification'],
            'model_used': MODEL_NAME
        }
        
    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': row.get('Job Title', ''),
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_NAME
        }

print("✅ Processing function defined")

## 11) Run Batch Processing

In [ ]:
# Process all job descriptions
results = []
print("🚀 Starting two-pass batch processing with self-consistency check...")
print(f"   Processing {len(df)} job descriptions\n")

# Track processing time
start_time = time.time()

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    
    # Periodic GPU memory cleanup
    if (idx + 1) % 10 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Small delay to prevent overheating
    time.sleep(0.1)

# Calculate processing time
elapsed_time = time.time() - start_time
avg_time_per_job = elapsed_time / len(df)

print(f"\n✅ Processed {len(results)} job descriptions")
print(f"   Total time: {elapsed_time:.2f} seconds")
print(f"   Average time per job: {avg_time_per_job:.2f} seconds")

# Save results
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / f"Job_Classifications_Batch_{MODEL_NAME.replace('/', '_')}_v754_two_pass.csv"
results_df.to_csv(output_path, index=False)
print(f"\n✅ Saved results to: {output_path}")

## 12) Generate Summary Statistics

In [ ]:
# Load the results
preds = results_df.copy()

# Generate summary statistics
major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

# Create summary
summary_stats = pd.DataFrame({
    'metric': ['total_rows', 'unique_major_roles', 'unique_minor_roles', 'specialist_count', 'executive_lead_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['major_role_group'].isin(EXECUTIVE_ROLES) & (preds['minor_sub_group'] == 'Lead')).sum())
    ]
})

summary_path = OUTPUTS_DIR / f"summary_stats_{MODEL_NAME.replace('/', '_')}_v754_two_pass.csv"
summary_stats.to_csv(summary_path, index=False)

# Show examples of classifications
examples = preds[['source_row_index', 'job_title_original', 'new_job_title', 
                  'major_role_group', 'minor_sub_group']].head(10)
examples_path = OUTPUTS_DIR / f"examples_{MODEL_NAME.replace('/', '_')}_v754_two_pass.csv"
examples.to_csv(examples_path, index=False)

print("\n📊 Summary Statistics:")
print(summary_stats.to_string(index=False))
print("\n📝 Major Role Distribution:")
print(major_counts.to_string())
print("\n📝 Minor Role Distribution:")
print(minor_counts.to_string())
print("\n📝 Example Classifications:")
print(examples.to_string(index=False))
print(f"\n✅ Saved summary to: {summary_path}")
print(f"✅ Saved examples to: {examples_path}")

## 13) Quality Check and Validation

In [ ]:
# Check for alignment issues between justification and selected roles
alignment_issues = []
for idx, row in preds.iterrows():
    justification = str(row['grouping_justification']).lower()
    major_role = str(row['major_role_group']).lower()
    # Check if justification mentions the selected role
    if major_role not in justification and major_role != 'other':
        alignment_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': row['major_role_group'],
            'justification_excerpt': row['grouping_justification'][:100] + '...'
        })

# Check for job title format consistency
title_format_issues = []
for idx, row in preds.iterrows():
    new_title = str(row['new_job_title'])
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    # Check if job title incorporates both major and minor roles
    if major_role.lower() not in new_title.lower() or minor_role.lower() not in new_title.lower():
        title_format_issues.append({
            'row_index': row['source_row_index'],
            'new_job_title': new_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role
        })

# Check for executive roles with "Lead" minor sub-grouping (should be rare)
executive_lead_issues = []
for idx, row in preds.iterrows():
    major_role = str(row['major_role_group'])
    minor_role = str(row['minor_sub_group'])
    if major_role in EXECUTIVE_ROLES and minor_role == 'Lead':
        executive_lead_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'new_job_title': row['new_job_title']
        })

# Save quality check results
if alignment_issues:
    alignment_df = pd.DataFrame(alignment_issues)
    alignment_path = OUTPUTS_DIR / f"alignment_issues_{MODEL_NAME.replace('/', '_')}_v754_two_pass.csv"
    alignment_df.to_csv(alignment_path, index=False)
    print(f"⚠️  Found {len(alignment_issues)} alignment issues - saved to {alignment_path}")
else:
    print("✅ No alignment issues found")

if title_format_issues:
    title_format_df = pd.DataFrame(title_format_issues)
    title_format_path = OUTPUTS_DIR / f"title_format_issues_{MODEL_NAME.replace('/', '_')}_v754_two_pass.csv"
    title_format_df.to_csv(title_format_path, index=False)
    print(f"⚠️  Found {len(title_format_issues)} title format issues - saved to {title_format_path}")
else:
    print("✅ No title format issues found")

if executive_lead_issues:
    executive_lead_df = pd.DataFrame(executive_lead_issues)
    executive_lead_path = OUTPUTS_DIR / f"executive_lead_issues_{MODEL_NAME.replace('/', '_')}_v754_two_pass.csv"
    executive_lead_df.to_csv(executive_lead_path, index=False)
    print(f"⚠️  Found {len(executive_lead_issues)} executive roles with 'Lead' minor sub-grouping - saved to {executive_lead_path}")
else:
    print("✅ No executive roles with inappropriate 'Lead' minor sub-grouping found")

print("\n✅ Enhanced quality check completed")

## 14) Cleanup and Final Report

In [ ]:
# Final GPU memory cleanup
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    final_allocated = torch.cuda.memory_allocated() / 1024**3
    final_reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"\n📊 Final GPU Memory: {final_allocated:.2f} GB allocated, {final_reserved:.2f} GB reserved")

print("\n" + "="*60)
print("✅ PROCESSING COMPLETE!")
print("="*60)
print(f"\n📁 All results saved to: {OUTPUTS_DIR}")
print(f"\n📄 Main output file: Job_Classifications_Batch_{MODEL_NAME.replace('/', '_')}_v754_two_pass.csv")
print(f"\n🎉 Job classification completed successfully!")